In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install dagshub
!pip install mlflow

In [3]:
import dagshub
dagshub.init(repo_owner='icosahedron31', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)



Accessing as icosahedron31

Initialized MLflow to track repo "icosahedron31/IEEE-CIS-Fraud-Detection"

Repository icosahedron31/IEEE-CIS-Fraud-Detection initialized!

# Reading Data

In [4]:
df_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")


In [5]:
df_identity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144233 entries, 0 to 144232
Data columns (total 41 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionID  144233 non-null  int64  
 1   id_01          144233 non-null  float64
 2   id_02          140872 non-null  float64
 3   id_03          66324 non-null   float64
 4   id_04          66324 non-null   float64
 5   id_05          136865 non-null  float64
 6   id_06          136865 non-null  float64
 7   id_07          5155 non-null    float64
 8   id_08          5155 non-null    float64
 9   id_09          74926 non-null   float64
 10  id_10          74926 non-null   float64
 11  id_11          140978 non-null  float64
 12  id_12          144233 non-null  object 
 13  id_13          127320 non-null  float64
 14  id_14          80044 non-null   float64
 15  id_15          140985 non-null  object 
 16  id_16          129340 non-null  object 
 17  id_17          139369 non-nul

In [6]:
df_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")

In [7]:
df_transaction.info(), df_transaction.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), object(14)
memory usage: 1.7+ GB


(None,
        TransactionID        isFraud  TransactionDT  TransactionAmt  \
 count   5.905400e+05  590540.000000   5.905400e+05   590540.000000   
 mean    3.282270e+06       0.034990   7.372311e+06      135.027176   
 std     1.704744e+05       0.183755   4.617224e+06      239.162522   
 min     2.987000e+06       0.000000   8.640000e+04        0.251000   
 25%     3.134635e+06       0.000000   3.027058e+06       43.321000   
 50%     3.282270e+06       0.000000   7.306528e+06       68.769000   
 75%     3.429904e+06       0.000000   1.124662e+07      125.000000   
 max     3.577539e+06       1.000000   1.581113e+07    31937.391000   
 
                card1          card2          card3          card5  \
 count  590540.000000  581607.000000  588975.000000  586281.000000   
 mean     9898.734658     362.555488     153.194925     199.278897   
 std      4901.170153     157.793246      11.336444      41.244453   
 min      1000.000000     100.000000     100.000000     100.000000   
 2

In [8]:
df_transaction.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
df_train = df_transaction.merge(df_identity, on='TransactionID', how='left')
df_train.columns

Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5',
       ...
       'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38',
       'DeviceType', 'DeviceInfo'],
      dtype='object', length=434)

In [10]:
df_train.head(20)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.500,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.000,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.000,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.000,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.000,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
5,2987005,0,86510,49.000,W,5937,555.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2987006,0,86522,159.000,W,12308,360.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2987007,0,86529,422.500,W,12695,490.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2987008,0,86535,15.000,H,2803,100.0,150.0,visa,226.0,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
9,2987009,0,86536,117.000,W,17399,111.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
X = df_train.drop(columns=['isFraud'])
y = df_train['isFraud']
X.shape, y.shape

((590540, 433), (590540,))

# Train/test Split


In [12]:
from sklearn.model_selection import train_test_split



X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_val.shape

((472432, 433), (118108, 433))

In [13]:
import mlflow
import mlflow.xgboost
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)


def run_experiment(
    model,
    run_name,
    experiment_name,
    X_train,
    X_val,
    y_train,
    y_val
):
    # Set experiment
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=run_name):

        # Train
        model.fit(X_train, y_train)

        # Predict
        probs = model.predict_proba(X_val)[:, 1]
        preds = model.predict(X_val)
        train_probs = model.predict_proba(X_train)[:, 1]
        train_auc = roc_auc_score(y_train, train_probs)
        # Metrics
        auc = roc_auc_score(y_val, probs)
        accuracy = accuracy_score(y_val, preds)
        recall = recall_score(y_val, preds)
        precision = precision_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)

        fraud_mask = (y_val == 1)
        nonfraud_mask = (y_val == 0)
    
        mlflow.log_metrics({
            "train auc" : train_auc, 
            "auc": auc,
            "accuracy": accuracy,
            "recall": recall,
            "precision": precision,
            "f1": f1,
            "fraud_mean_prob": probs[fraud_mask].mean(),
            "nonfraud_mean_prob": probs[nonfraud_mask].mean()
        })

        # Log model parameters if available
        if hasattr(model, "get_params"):
            params = {k: str(v) for k, v in model.get_params().items()}
            mlflow.log_params(params)


        
        fpr, tpr, _ = roc_curve(y_val, probs)

        plt.figure()
        plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend()

        mlflow.log_figure(plt.gcf(), "roc_curve.png")
        plt.close()

        precision_vals, recall_vals, _ = precision_recall_curve(y_val, probs)

        plt.figure()
        plt.plot(recall_vals, precision_vals)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curve")

        mlflow.log_figure(plt.gcf(), "pr_curve.png")
        plt.close()

      
        cm = confusion_matrix(y_val, preds)

        plt.figure()
        ConfusionMatrixDisplay(cm).plot()
        plt.title("Confusion Matrix")

        mlflow.log_figure(plt.gcf(), "confusion_matrix.png")
        plt.close()

       
        plt.figure()

        plt.hist(
            probs[nonfraud_mask],
            bins=50,
            alpha=0.5,
            label="non_fraud"
        )

        plt.hist(
            probs[fraud_mask],
            bins=50,
            alpha=0.5,
            label="fraud"
        )

        plt.xlabel("Predicted Probability")
        plt.ylabel("Count")
        plt.title("Prediction Score Distribution")
        plt.legend()

        mlflow.log_figure(
            plt.gcf(),
            "score_distribution.png"
        )
        plt.close()

        # Log model artifact
        mlflow.sklearn.log_model(model, "model")

        return {
            "auc": auc,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

# Preprocessing

In [14]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class MissingValueFilter(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        column_threshold=0.9,
        row_threshold=None
    ):
        self.column_threshold = column_threshold
        self.row_threshold = row_threshold

    def fit(self, X, y=None):

        missing_ratio = X.isnull().mean()

        self.columns_to_keep_ = missing_ratio[
            missing_ratio <= self.column_threshold
        ].index.tolist()

        return self

    def transform(self, X):

        X = X.copy()

        X = X[self.columns_to_keep_]

        if self.row_threshold is not None:

            row_missing_ratio = X.isnull().mean(axis=1)

            X = X[
                row_missing_ratio <= self.row_threshold
            ]

        return X

In [15]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class NAPreprocessor(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        numeric_strategy="median",
        categorical_strategy="mode"
    ):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy

    def fit(self, X, y=None):
        X = X.copy()

        self.fill_values_ = {}

        for col in X.columns:

            # Numeric columns
            if pd.api.types.is_numeric_dtype(X[col]):

                if self.numeric_strategy == "mean":
                    value = X[col].mean()

                elif self.numeric_strategy == "median":
                    value = X[col].median()

             

            # Categorical columns
            else:

                if self.categorical_strategy == "mode":

                    mode_vals = X[col].mode()

                    if len(mode_vals) > 0:
                        value = mode_vals.iloc[0]
                    else:
                        value = "missing"

                elif self.categorical_strategy == "constant":
                    value = "missing"


            self.fill_values_[col] = value

        return self

    def transform(self, X):
        X = X.copy()

        for col, fill_value in self.fill_values_.items():

            if col in X.columns:
                X[col] = X[col].fillna(fill_value)

        return X

In [16]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder

class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.cat_cols = None
        self.num_cols = None
        self.encoder = None
        self.medians = None

    def fit(self, X, y=None):
        X = X.copy()
        self.cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
        self.num_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
        self.medians = X[self.num_cols].median()
        if self.cat_cols:
            X[self.cat_cols] = X[self.cat_cols].fillna("missing")
            self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            self.encoder.fit(X[self.cat_cols])
        return self

    def transform(self, X):
        X = X.copy()
        if self.cat_cols:
            X[self.cat_cols] = X[self.cat_cols].fillna("missing")
            if self.encoder is not None:
                X[self.cat_cols] = self.encoder.transform(X[self.cat_cols])
        for col in self.num_cols:
            if col in X.columns:
                X[col] = X[col].fillna(self.medians[col])
        for col in X.columns:
            X[col] = pd.to_numeric(X[col], errors="coerce")
        X = X.fillna(0)
        return X

In [17]:
import pandas as pd
import numpy as np

class WOEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None, smoothing=0.5):
        self.cols = cols
        self.smoothing = smoothing
        self.woe_maps = {}

    def fit(self, X, y):
        X = X.copy()
        cols = self.cols or X.select_dtypes(include=["object", "category"]).columns.tolist()
        
        total_events = y.sum()
        total_non_events = (1 - y).sum()

        for col in cols:
            tmp = pd.DataFrame({"col": X[col], "target": y.values})
            stats = tmp.groupby("col")["target"].agg(["sum", "count"])
            stats.columns = ["events", "count"]
            stats["non_events"] = stats["count"] - stats["events"]

            # smoothing to avoid log(0)
            stats["dist_events"]     = (stats["events"] + self.smoothing) / (total_events + self.smoothing)
            stats["dist_non_events"] = (stats["non_events"] + self.smoothing) / (total_non_events + self.smoothing)
            stats["woe"] = np.log(stats["dist_events"] / stats["dist_non_events"])

            self.woe_maps[col] = stats["woe"].to_dict()

        return self

    def transform(self, X):
        X = X.copy()
        for col, woe_map in self.woe_maps.items():
            if col in X.columns:
                X[col] = X[col].map(woe_map).fillna(0)  # unseen → 0 (neutral)
        return X

In [20]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        # Sample if too large
        if len(X) > 5000:
            X_sample = X.sample(5000, random_state=42)
        else:
            X_sample = X
            
        corr_matrix = X_sample.corr().abs()  # faster on sample
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.cols_to_drop_ = [col for col in upper.columns if any(upper[col] > 0.8)]
        print(self.cols_to_drop_)
        return self

    def transform(self, X):
        return pd.DataFrame(X).drop(columns=self.cols_to_drop_, errors='ignore')

In [19]:
woe_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
woe_cols

['ProductCD',
 'card4',
 'card6',
 'P_emaildomain',
 'R_emaildomain',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'id_12',
 'id_15',
 'id_16',
 'id_23',
 'id_27',
 'id_28',
 'id_29',
 'id_30',
 'id_31',
 'id_33',
 'id_34',
 'id_35',
 'id_36',
 'id_37',
 'id_38',
 'DeviceType',
 'DeviceInfo']

# Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
pipeline_lr = Pipeline([
    
    ("preprocess", FraudPreprocessor()),
    ("woe", WOEEncoder(cols=woe_cols)),
    ("scaler", StandardScaler()),
    ("correlation", CorrelationFilter()),
    ("model", LogisticRegression(
      
        class_weight="balanced",
        max_iter=10000,
        solver="saga",
        random_state=42
    ))
])

run_experiment(pipeline_lr, "Logistic Regression WOE", "Logistic Regression",
               X_train, X_val, y_train, y_val)

In [ ]:
run_experiment(pipeline, "Preprocessing Missing values 97%", "Preprocessing", X_train, X_val, y_train, y_val)

# Handling Imbalanced 

Undersampling

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline  # not sklearn

pipeline_lr = Pipeline([
    ("preprocess", FraudPreprocessor()),
    ("woe", WOEEncoder(cols=woe_cols)),
    ("corr_filter", CorrelationFilter()),
    ("scaler", StandardScaler()),
    ("undersample", RandomUnderSampler(random_state=42)),
    ("model", LogisticRegression(
        max_iter=1000,
        solver="saga",
        random_state=42
    ))
])

run_experiment(pipeline_lr, "LR Undersample", "Logistic Regression",
               X_train, X_val, y_train, y_val)

In [ ]:
run_experiment(pipeline, "Undersampling", "Handling Imbalanced", X_train, X_val, y_train, y_val)

# Feature Engineering 

In [21]:
import pandas as pd
import numpy as np

def engineer_features(X, y=None):
    X = X.copy()

    X['amt_log'] = np.log1p(X['TransactionAmt'])
    X['amt_rounded'] = (X['TransactionAmt'] % 1 == 0).astype(int)  # whole number flag
    

    X['hour'] = (X['TransactionDT'] // 3600) % 24
    X['day_of_week'] = (X['TransactionDT'] // (3600 * 24)) % 7
    X['is_weekend'] = X['day_of_week'].isin([5, 6]).astype(int)

    X['email_match'] = (X['P_emaildomain'] == X['R_emaildomain']).astype(int)
    
    # --- Aggregates per card ---
    for col in ['card1', 'card2', 'card4', 'card6']:
        if col in X.columns:
            X[f'{col}_txn_count'] = X.groupby(col)['TransactionAmt'].transform('count')
            X[f'{col}_amt_mean']  = X.groupby(col)['TransactionAmt'].transform('mean')
            X[f'{col}_amt_std']   = X.groupby(col)['TransactionAmt'].transform('std')
            X[f'{col}_amt_dev']   = X['TransactionAmt'] - X[f'{col}_amt_mean']
            X[f'{col}_amt_zscore'] = X[f'{col}_amt_dev'] / (X[f'{col}_amt_std'] + 1e-9)

    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in X.columns:
            X[f'{col}_freq'] = X.groupby(col)['TransactionAmt'].transform('count')

    if 'DeviceInfo' in X.columns:
        X['device_freq'] = X.groupby('DeviceInfo')['TransactionAmt'].transform('count')

    if y is not None:
        tmp = X.copy()
        tmp['target'] = y.values
        for col in ['card4', 'card6', 'P_emaildomain', 'DeviceType']:
            if col in X.columns:
                fraud_rate = tmp.groupby(col)['target'].transform('mean')
                X[f'{col}_fraud_rate'] = fraud_rate

    # --- Time since last transaction per card ---
    X = X.sort_values('TransactionDT')
    X['time_since_last_txn'] = X.groupby('card1')['TransactionDT'].diff().fillna(0)

    return X


# Usage
X_engineered = engineer_features(X, y)

In [22]:
from sklearn.model_selection import train_test_split



X_train_eng, X_val_eng, y_train, y_val = train_test_split(
    X_engineered, y,
    test_size=0.2,
    random_state=42
)

X_train_eng.shape, X_val_eng.shape


((472432, 467), (118108, 467))

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
woe_cols = X_train_eng.select_dtypes(include=["object", "category"]).columns.tolist()
print(woe_cols)
pipeline_lr = Pipeline([
   
    ("preprocess", FraudPreprocessor()),
    ("woe", WOEEncoder(cols=woe_cols)),
    ("scaler", StandardScaler()),
    ("correlation", CorrelationFilter()),
    ("model", LogisticRegression(
      
        class_weight="balanced",
        max_iter=1050,
        solver="saga",
        random_state=42
    ))
])


['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [24]:
run_experiment(pipeline_lr, "Feature Engineering", "Logistic Regression", X_train_eng, X_val_eng, y_train, y_val)

[1, 17, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 46, 47, 50, 52, 53, 58, 62, 64, 66, 68, 69, 70, 71, 73, 74, 75, 79, 81, 83, 84, 85, 86, 87, 89, 91, 93, 95, 96, 97, 98, 100, 102, 103, 104, 105, 107, 110, 111, 112, 113, 115, 116, 117, 120, 122, 123, 124, 125, 126, 127, 129, 132, 133, 134, 136, 137, 138, 140, 143, 144, 145, 146, 147, 149, 150, 153, 154, 155, 156, 157, 158, 159, 163, 165, 166, 167, 169, 172, 178, 179, 180, 181, 185, 186, 187, 189, 190, 193, 194, 195, 196, 198, 200, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 230, 231, 232, 233, 234, 235, 236, 238, 239, 242, 243, 244, 245, 246, 248, 249, 250, 251, 252, 254, 255, 256, 257, 260, 262, 263, 264, 265, 266, 267, 268, 269, 271, 272, 275, 278, 283, 284, 285, 286, 287, 289, 290, 291, 292, 296, 297, 299, 300, 301, 302, 304, 305, 306, 307, 308, 309, 310, 311, 312, 316, 318, 319, 321, 322, 324, 325, 326, 327, 328, 331, 332, 333, 337, 338, 340, 342, 346, 347, 348, 349, 350, 351

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
2026/05/04 13:45:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 13:45:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Feature Engineering at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/6/runs/0533c2d270f446dea2aa03a440e50bd7
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/6


{'auc': np.float64(0.8495502745383576),
 'accuracy': 0.8183611609713144,
 'precision': 0.1309345113007677,
 'recall': 0.7197076850542197,
 'f1': 0.22156101455060054}

<Figure size 640x480 with 0 Axes>

# Dropping some features

In [ ]:
print(X_train_eng.shape)
pipeline.fit(X_train_eng, y_train)

importances = pd.Series(
    pipeline['model'].feature_importances_,
    index=X_train_eng.columns 
)

threshold = importances.quantile(0.2)
drop_cols = importances[importances <= threshold].index.tolist()
print(f"Dropping {len(drop_cols)} columns: {drop_cols}")

X_train_reduced = X_train_eng.drop(columns=drop_cols)
X_val_reduced   = X_val_eng.drop(columns=drop_cols)

run_experiment(pipeline, "Drop Features 20% bigger model", "", X_train_reduced, X_val_reduced, y_train, y_val)